
1. Escribir src/preprocessing.py


In [1]:
%%writefile ../src/preprocessing.py
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np
import pandas as pd


# ──────────────────────────────────────────────
# ROL 2 — Winsorizing por IQR
# ──────────────────────────────────────────────

class IQRWinsorizor(BaseEstimator, TransformerMixin):
    """
    Trata outliers aplicando capping por IQR.
    fit() calcula los límites; transform() aplica el clip.
    """
    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):
        X = np.array(X)
        q1 = np.percentile(X, 25, axis=0)
        q3 = np.percentile(X, 75, axis=0)
        iqr = q3 - q1
        self.lower_ = q1 - self.factor * iqr
        self.upper_ = q3 + self.factor * iqr
        return self

    def transform(self, X, y=None):
        X = np.array(X, dtype=float).copy()
        for i in range(X.shape[1]):
            X[:, i] = np.clip(X[:, i], self.lower_[i], self.upper_[i])
        return X


# ──────────────────────────────────────────────
# ROL 3 — Feature Engineering
# ──────────────────────────────────────────────

class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Crea features derivadas a partir de las columnas numéricas originales.

    Features nuevas:
    - credit_utilization : avg_bill / LIMIT_BAL
    - payment_ratio      : avg_pay / (avg_bill + 1)
    - late_payments      : meses con atraso (PAY_2..PAY_6 > 0)
    - bill_trend         : BILL_AMT6 - BILL_AMT1
    - pay_trend          : PAY_AMT6 - PAY_AMT1
    - risk_score         : LIMIT_BAL * late_payments
    - ability_to_pay     : PAY_AMT1 + PAY_AMT2 + PAY_AMT3
    - log_limit          : log1p(LIMIT_BAL)
    - log_avg_bill       : log1p(avg_bill clipped >= 0)
    - age_group          : binning de AGE en 4 grupos
    """

    NUM_COLS = [
        "LIMIT_BAL", "AGE",
        "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
        "BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
        "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6",
    ]

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        df = pd.DataFrame(X, columns=self.NUM_COLS)

        avg_bill = df[[f"BILL_AMT{i}" for i in range(1, 7)]].mean(axis=1)
        avg_pay  = df[[f"PAY_AMT{i}" for i in range(1, 7)]].mean(axis=1)

        df["credit_utilization"] = avg_bill / df["LIMIT_BAL"]
        df["payment_ratio"]      = avg_pay / (avg_bill + 1)
        df["payment_ratio"]      = df["payment_ratio"].fillna(df["payment_ratio"].median())

        pay_cols = ["PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
        df["late_payments"]  = df[pay_cols].apply(lambda r: (r > 0).sum(), axis=1)
        df["bill_trend"]     = df["BILL_AMT6"] - df["BILL_AMT1"]
        df["pay_trend"]      = df["PAY_AMT6"]  - df["PAY_AMT1"]
        df["risk_score"]     = df["LIMIT_BAL"] * df["late_payments"]
        df["ability_to_pay"] = df["PAY_AMT1"]  + df["PAY_AMT2"] + df["PAY_AMT3"]
        df["log_limit"]      = np.log1p(df["LIMIT_BAL"])
        df["log_avg_bill"]   = np.log1p(avg_bill.clip(lower=0))
        df["age_group"]      = pd.cut(
            df["AGE"], bins=[0, 25, 35, 50, 100], labels=[0, 1, 2, 3]
        ).astype(float)

        return df.values


# ──────────────────────────────────────────────
# Función principal
# ──────────────────────────────────────────────

def build_preprocessor(num_cols, cat_cols):
    """
    Construye el ColumnTransformer con:
    - num_pipe: Imputación + Winsorizing IQR + FeatureEngineering + StandardScaler
    - cat_pipe: Imputación + OneHotEncoder
    """
    num_pipe = Pipeline(steps=[
        ("imp",      SimpleImputer(strategy="median")),
        ("outliers", IQRWinsorizor(factor=1.5)),
        ("features", FeatureEngineer()),
        ("sc",       StandardScaler()),
    ])

    cat_pipe = Pipeline(steps=[
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh",  OneHotEncoder(handle_unknown="ignore")),
    ])

    preproc = ColumnTransformer(transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])

    return preproc

Overwriting ../src/preprocessing.py


Verificamos que exista la carpeta sr

In [3]:
import os
print(os.listdir("../src"))

['preprocessing.py', '__pycache__']




 3. Cargar dataset raw y hacer train/test split

> ⚠️ Se carga el dataset **raw** (no el ya procesado) para que el pipeline aplique todas las transformaciones correctamente desde el inicio.
>
> El balanceo (Rol 4) se aplica **solo al train set**.


In [2]:
import pandas as pd
import sys
sys.path.append("..")

from sklearn.model_selection import train_test_split
from src.preprocessing import build_preprocessor

df = pd.read_csv('../data/raw/04-default_credit.csv')

TARGET = "default payment next month"
X = df.drop(columns=[TARGET, "ID"])
y = df[TARGET]

print("Shape:", X.shape)
print("Distribución target:")
print(y.value_counts(normalize=True).round(4))

Shape: (30000, 23)
Distribución target:
default payment next month
0    0.7788
1    0.2212
Name: proportion, dtype: float64


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

Train: 24000 | Test: 6000



4. Aplicar ColumnTransformer y validar fit/transform


In [4]:
# Detectar columnas
num_cols = [
    "LIMIT_BAL", "AGE",
    "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6",
]

cat_cols = ["SEX", "EDUCATION", "MARRIAGE"]

print("Numéricas:", len(num_cols), num_cols)
print("Categóricas:", len(cat_cols), cat_cols)

Numéricas: 20 ['LIMIT_BAL', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
Categóricas: 3 ['SEX', 'EDUCATION', 'MARRIAGE']


In [5]:
preproc = build_preprocessor(num_cols, cat_cols)

# fit SOLO con train
preproc.fit(X_train)

X_train_transformed = preproc.transform(X_train)
X_test_transformed  = preproc.transform(X_test)

print("X_train transformado:", X_train_transformed.shape)
print("X_test  transformado:", X_test_transformed.shape)
print("✓ Pipeline fit/transform sin errores")

X_train transformado: (24000, 43)
X_test  transformado: (6000, 43)
✓ Pipeline fit/transform sin errores



5. Rol 4 — Balanceo solo en train set

Según el análisis del Rol 4, la clase minoritaria (~22%) está por debajo del umbral del 30%.  
Las mejores técnicas fueron **RandomOverSampler** y **RandomUnderSampler**.


In [8]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import os

print("Distribución ANTES del balanceo:")
print(y_train.value_counts())

samplers = {
    "RandomOverSampler":  RandomOverSampler(random_state=42),
    "RandomUnderSampler": RandomUnderSampler(random_state=42),
}

os.makedirs('../data/processed', exist_ok=True)

for name, sampler in samplers.items():
    X_res, y_res = sampler.fit_resample(X_train_transformed, y_train)

    df_res = pd.DataFrame(X_res)
    df_res[TARGET] = y_res.values
    df_res.to_csv(f'../data/processed/04_default_credit_balanced_{name}.csv', index=False)

    print(f"\n{name}: {X_res.shape[0]} muestras | {pd.Series(y_res).value_counts().to_dict()}")

# Guardar test sin balancear
df_test = pd.DataFrame(X_test_transformed)
df_test[TARGET] = y_test.values
df_test.to_csv('../data/processed/dataset_pipeline.csv', index=False)
print("\n✓ Test set guardado en data/processed/dataset_pipeline.csv")

Distribución ANTES del balanceo:
default payment next month
0    18691
1     5309
Name: count, dtype: int64

RandomOverSampler: 37382 muestras | {0: 18691, 1: 18691}

RandomUnderSampler: 10618 muestras | {0: 5309, 1: 5309}

✓ Test set guardado en data/processed/dataset_pipeline.csv



6. Persistencia del pipeline con joblib


In [9]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

joblib.dump(preproc, '../models/preprocessor.pkl')
print("✓ Pipeline guardado en models/preprocessor.pkl")

# Verificar carga
preproc_loaded = joblib.load('../models/preprocessor.pkl')
X_verify = preproc_loaded.transform(X_test)
assert X_verify.shape == X_test_transformed.shape
print("✓ Pipeline cargado y verificado correctamente")

✓ Pipeline guardado en models/preprocessor.pkl
✓ Pipeline cargado y verificado correctamente
